# Load your dataset to Argilla

In [1]:
!pip install -q argilla datasets

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.3/161.3 kB 4.9 MB/s eta 0:00:00


In [2]:
import argilla as rg

ARGILLA_API_URL = "https://parthgeek-argilla-parth-test.hf.space/"
ARGILLA_API_KEY = "owner.apikey"

print("=" * 80)
print("Connecting to Argilla...")
print("=" * 80)

client = rg.Argilla(
    api_url=ARGILLA_API_URL,
    api_key=ARGILLA_API_KEY,
)

print("Connected to Argilla successfully.")
print(client)
print("=" * 80)

/usr/local/lib/python3.12/dist-packages/argilla/_api/_token.py:83: UserWarning: 
The secrets ARGILLA_API_URL and does not exist in your Colab secrets.
  warnings.warn(f"\nThe secrets {name} and does not exist in your Colab secrets.")
/usr/local/lib/python3.12/dist-packages/argilla/_api/_token.py:83: UserWarning: 
The secrets ARGILLA_API_KEY and does not exist in your Colab secrets.
  warnings.warn(f"\nThe secrets {name} and does not exist in your Colab secrets.")


Connecting to Argilla...
Connected to Argilla successfully.


Argilla has been deployed at: https://parthgeek-argilla-parth-test.hf.space/


In [3]:
from datasets import load_dataset

print("=" * 80)
print("Loading SetFit/ag_news dataset...")
print("=" * 80)

data = load_dataset("SetFit/ag_news", split="train[:100]")

print("Dataset loaded successfully.")
print(data)
print("Features:")
print(data.features)
print("Unique labels:")
print(data.unique("label_text"))
print("=" * 80)

Loading SetFit/ag_news dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


train.jsonl:   0%|          | 0.00/33.8M [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/2.13M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

Dataset loaded successfully.
Dataset({
    features: ['text', 'label', 'label_text'],
    num_rows: 100
})
Features:
{'text': Value('string'), 'label': Value('int64'), 'label_text': Value('string')}
Unique labels:
['Business', 'Sci/Tech']


In [4]:
print("=" * 80)
print("Creating Argilla dataset settings...")
print("=" * 80)

settings = rg.Settings(
    guidelines="Classify the news text and highlight named entities if present.",
    fields=[
        rg.TextField(
            name="text",
            title="News Text"
        )
    ],
    questions=[
        rg.LabelQuestion(
            name="label",
            title="Classify the text:",
            labels=data.unique("label_text")
        ),
        rg.SpanQuestion(
            name="entities",
            title="Highlight all the entities in the text:",
            labels=["PERSON", "ORG", "LOC", "EVENT"],
            field="text",
        ),
    ],
)

print("Settings created successfully.")
print("=" * 80)

Creating Argilla dataset settings...
Settings created successfully.


In [5]:
DATASET_NAME = "ag_news_setfit"

print("=" * 80)
print("Creating dataset in Argilla...")
print("Dataset name:", DATASET_NAME)
print("=" * 80)

existing_dataset = client.datasets(name=DATASET_NAME)

if existing_dataset is not None:
    print("Dataset already exists:", DATASET_NAME)
    dataset = existing_dataset
else:
    dataset = rg.Dataset(
        name=DATASET_NAME,
        settings=settings,
        client=client
    )

    dataset.create()
    print("Dataset created successfully:", dataset.name)

print("=" * 80)

Creating dataset in Argilla...
Dataset name: ag_news_setfit


/usr/local/lib/python3.12/dist-packages/argilla/client.py:356: UserWarning: Dataset with name 'ag_news_setfit' not found in workspace 'argilla'
  warnings.warn(f"Dataset with name {name!r} not found in workspace {workspace.name!r}")
/usr/local/lib/python3.12/dist-packages/argilla/datasets/_resource.py:264: UserWarning: Workspace not provided. Using default workspace: argilla id: 7be4e061-4be5-4d54-9d9f-76712504137e
  warnings.warn(f"Workspace not provided. Using default workspace: {workspace.name} id: {workspace.id}")


Dataset created successfully: ag_news_setfit


In [6]:
print("=" * 80)
print("Preparing records for Argilla...")
print("=" * 80)

records = [
    rg.Record(
        fields={
            "text": row["text"]
        },
        suggestions=[
            rg.Suggestion(
                question_name="label",
                value=row["label_text"]
            )
        ]
    )
    for row in data
]

print("Number of records prepared:", len(records))
print("=" * 80)

print("Logging records to Argilla...")

dataset.records.log(records)

print("=" * 80)
print("Records logged successfully.")
print("Open Argilla here:")
print(ARGILLA_API_URL)
print("Dataset name:", DATASET_NAME)
print("=" * 80)

Preparing records for Argilla...
Number of records prepared: 100
Logging records to Argilla...


DatasetRecords: The provided batch size 256 was normalized. Using value 100.

Sending records...: 100%|██████████| 1/1 [00:04<00:00,  4.89s/batch]

Records logged successfully.
Open Argilla here:
https://parthgeek-argilla-parth-test.hf.space/
Dataset name: ag_news_setfit
